In [3]:
import requests
import pandas as pd

API_URL = f"https://tabular-api.data.gouv.fr/api/resources/17af574e-ce76-4092-a1f5-f8bbffa307ee/data/"
df_election = pd.read_csv('src/election.csv', sep=';')

all_data = []
page = 1

while True:
    response = requests.get(API_URL, params={"page": page, "page_size": 200})
    result = response.json()
    all_data.extend(result["data"])

    if result["links"]["next"] is None:
        break
    page += 1

df = pd.DataFrame(all_data)
display(df)

/tmp/ipykernel_14343/2043197130.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_election = pd.read_csv('src/election.csv', sep=';')


,__id,CodeInsee,Inscrits,Abstentions,Votants,Blancs,Exprimés,Joly,LePen,Sarkozy,Mélenchon,Poutou,Arthaud,Cheminade,Bayrou,DupontAignan,Hollande
0,1,01001,592,84,508,9,499,13,126,159,25,2,2,2,54,4,112
1,2,01002,215,36,179,5,174,10,38,45,15,2,0,1,22,2,39
2,3,01004,8205,1698,6507,126,6381,124,1359,1603,778,83,41,21,543,138,1691
3,4,01005,1152,170,982,18,964,16,238,323,57,6,6,0,97,26,195
4,5,01006,105,17,88,1,87,3,25,19,9,1,1,0,8,0,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36786,36787,ZZ007,89643,54981,34662,318,34344,3473,1446,10363,2592,248,97,138,4911,431,10645
36787,36788,ZZ008,109763,84466,25297,291,25006,1011,1321,12256,1926,203,112,92,1722,315,6048
36788,36789,ZZ009,98997,59887,39110,321,38789,1010,1395,12307,4378,209,78,87,2615,269,16441
36789,36790,ZZ010,89859,46782,43077,566,42511,1320,3254,17882,2484,206,101,147,3779,441,12897


In [4]:
df['gauche'] = df['Hollande'] + df['Mélenchon'] + df['Joly'] + df['Arthaud'] + df['Poutou']
df['centre'] = df['Bayrou']
df['droite'] = df['Sarkozy'] + df['LePen'] + df['DupontAignan'] + df['Cheminade']

df = df.drop(columns=['Hollande', 'Mélenchon', 'Joly', 'Arthaud', 'Poutou', 'Bayrou', 'Sarkozy', 'LePen', 'DupontAignan', 'Cheminade'])

df['Année'] = 2012

df = df.rename(columns={'CodeInsee': 'Code_INSEE'})

display(df)

,__id,Code_INSEE,Inscrits,Abstentions,Votants,Blancs,Exprimés,gauche,centre,droite,Année
0,1,01001,592,84,508,9,499,154,54,291,2012
1,2,01002,215,36,179,5,174,66,22,86,2012
2,3,01004,8205,1698,6507,126,6381,2717,543,3121,2012
3,4,01005,1152,170,982,18,964,280,97,587,2012
4,5,01006,105,17,88,1,87,35,8,44,2012
...,...,...,...,...,...,...,...,...,...,...,...
36786,36787,ZZ007,89643,54981,34662,318,34344,17055,4911,12378,2012
36787,36788,ZZ008,109763,84466,25297,291,25006,9300,1722,13984,2012
36788,36789,ZZ009,98997,59887,39110,321,38789,22116,2615,14058,2012
36789,36790,ZZ010,89859,46782,43077,566,42511,17008,3779,21724,2012


In [5]:
df['% Abs/Ins'] = df['Abstentions'] / df['Inscrits'] * 100
df['% Vot/Ins'] = df['Votants'] / df['Inscrits'] * 100

df['% Blancs/Ins'] = df['Blancs'] / df['Inscrits'] * 100
df['% Blancs/Vot'] = df['Blancs'] / df['Votants'] * 100

df['Nuls'] = 0 
df['% Nuls/Ins'] = 0
df['% Nuls/Vot'] = 0

df['% Exp/Ins'] = df['Exprimés'] / df['Inscrits'] * 100
df['% Exp/Vot'] = df['Exprimés'] / df['Votants'] * 100

df['% gauche/Exp'] = df['gauche'] / df['Exprimés'] * 100
df['% centre/Exp'] = df['centre'] / df['Exprimés'] * 100
df['% droite/Exp'] = df['droite'] / df['Exprimés'] * 100

df = df.drop(columns=['gauche', 'centre', 'droite', '__id'])
display(df)

,Code_INSEE,Inscrits,Abstentions,Votants,Blancs,Exprimés,Année,% Abs/Ins,% Vot/Ins,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp
0,01001,592,84,508,9,499,2012,14.189189,85.810811,1.520270,1.771654,0,0,0,84.290541,98.228346,30.861723,10.821643,58.316633
1,01002,215,36,179,5,174,2012,16.744186,83.255814,2.325581,2.793296,0,0,0,80.930233,97.206704,37.931034,12.643678,49.425287
2,01004,8205,1698,6507,126,6381,2012,20.694698,79.305302,1.535649,1.936376,0,0,0,77.769653,98.063624,42.579533,8.509638,48.910829
3,01005,1152,170,982,18,964,2012,14.756944,85.243056,1.562500,1.832994,0,0,0,83.680556,98.167006,29.045643,10.062241,60.892116
4,01006,105,17,88,1,87,2012,16.190476,83.809524,0.952381,1.136364,0,0,0,82.857143,98.863636,40.229885,9.195402,50.574713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36786,ZZ007,89643,54981,34662,318,34344,2012,61.333289,38.666711,0.354740,0.917431,0,0,0,38.311971,99.082569,49.659329,14.299441,36.041230
36787,ZZ008,109763,84466,25297,291,25006,2012,76.953072,23.046928,0.265117,1.150334,0,0,0,22.781812,98.849666,37.191074,6.886347,55.922579
36788,ZZ009,98997,59887,39110,321,38789,2012,60.493752,39.506248,0.324252,0.820762,0,0,0,39.181995,99.179238,57.016164,6.741602,36.242234
36789,ZZ010,89859,46782,43077,566,42511,2012,52.061563,47.938437,0.629876,1.313926,0,0,0,47.308561,98.686074,40.008468,8.889464,51.102068


In [6]:
cols_groups = ['% gauche/Exp', '% centre/Exp', '% droite/Exp']
mask = df[cols_groups].notna().any(axis=1)
df.loc[mask, 'Résultat'] = df.loc[mask, cols_groups].idxmax(axis=1)

groups_dict = {'% gauche/Exp': 'gauche', '% centre/Exp': "centre", '% droite/Exp': 'droite'}
df['Résultat'] = df['Résultat'].replace(groups_dict)


display(df)

,Code_INSEE,Inscrits,Abstentions,Votants,Blancs,Exprimés,Année,% Abs/Ins,% Vot/Ins,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Résultat
0,01001,592,84,508,9,499,2012,14.189189,85.810811,1.520270,1.771654,0,0,0,84.290541,98.228346,30.861723,10.821643,58.316633,droite
1,01002,215,36,179,5,174,2012,16.744186,83.255814,2.325581,2.793296,0,0,0,80.930233,97.206704,37.931034,12.643678,49.425287,droite
2,01004,8205,1698,6507,126,6381,2012,20.694698,79.305302,1.535649,1.936376,0,0,0,77.769653,98.063624,42.579533,8.509638,48.910829,droite
3,01005,1152,170,982,18,964,2012,14.756944,85.243056,1.562500,1.832994,0,0,0,83.680556,98.167006,29.045643,10.062241,60.892116,droite
4,01006,105,17,88,1,87,2012,16.190476,83.809524,0.952381,1.136364,0,0,0,82.857143,98.863636,40.229885,9.195402,50.574713,droite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36786,ZZ007,89643,54981,34662,318,34344,2012,61.333289,38.666711,0.354740,0.917431,0,0,0,38.311971,99.082569,49.659329,14.299441,36.041230,gauche
36787,ZZ008,109763,84466,25297,291,25006,2012,76.953072,23.046928,0.265117,1.150334,0,0,0,22.781812,98.849666,37.191074,6.886347,55.922579,droite
36788,ZZ009,98997,59887,39110,321,38789,2012,60.493752,39.506248,0.324252,0.820762,0,0,0,39.181995,99.179238,57.016164,6.741602,36.242234,gauche
36789,ZZ010,89859,46782,43077,566,42511,2012,52.061563,47.938437,0.629876,1.313926,0,0,0,47.308561,98.686074,40.008468,8.889464,51.102068,droite


In [7]:
df = df.merge(df_election[['Code_INSEE', 'Libellé de la commune']], on='Code_INSEE', how='left')
df = df.dropna()

In [8]:
display(df)

,Code_INSEE,Inscrits,Abstentions,Votants,Blancs,Exprimés,Année,% Abs/Ins,% Vot/Ins,% Blancs/Ins,...,Nuls,% Nuls/Ins,% Nuls/Vot,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Résultat,Libellé de la commune
0,01001,592,84,508,9,499,2012,14.189189,85.810811,1.520270,...,0,0,0,84.290541,98.228346,30.861723,10.821643,58.316633,droite,L'Abergement-Clémenciat
1,01001,592,84,508,9,499,2012,14.189189,85.810811,1.520270,...,0,0,0,84.290541,98.228346,30.861723,10.821643,58.316633,droite,L'Abergement-Clémenciat
2,01002,215,36,179,5,174,2012,16.744186,83.255814,2.325581,...,0,0,0,80.930233,97.206704,37.931034,12.643678,49.425287,droite,L'Abergement-de-Varey
3,01002,215,36,179,5,174,2012,16.744186,83.255814,2.325581,...,0,0,0,80.930233,97.206704,37.931034,12.643678,49.425287,droite,L'Abergement-de-Varey
4,01004,8205,1698,6507,126,6381,2012,20.694698,79.305302,1.535649,...,0,0,0,77.769653,98.063624,42.579533,8.509638,48.910829,droite,Ambérieu-en-Bugey
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55380,ZZ009,98997,59887,39110,321,38789,2012,60.493752,39.506248,0.324252,...,0,0,0,39.181995,99.179238,57.016164,6.741602,36.242234,gauche,Alger
55381,ZZ010,89859,46782,43077,566,42511,2012,52.061563,47.938437,0.629876,...,0,0,0,47.308561,98.686074,40.008468,8.889464,51.102068,droite,Almaty
55382,ZZ010,89859,46782,43077,566,42511,2012,52.061563,47.938437,0.629876,...,0,0,0,47.308561,98.686074,40.008468,8.889464,51.102068,droite,Almaty
55383,ZZ011,80061,42911,37150,488,36662,2012,53.597882,46.402118,0.609535,...,0,0,0,45.792583,98.686406,38.118488,12.806175,49.075337,droite,Amman


In [ ]:
df_election = pd.concat([df_election, df], ignore_index=True)
df_election['Code_INSEE'] = df_election['Code_INSEE'].astype(str)
df_election = df_election.sort_values(by=['Code_INSEE']).reset_index(drop=True)
df_election = df_election.drop_duplicates(subset=['Code_INSEE', 'Année']).reset_index(drop=True)
display(df_election)

df_election.to_csv('src/election.csv', index=False, sep=';', encoding='utf-8')

,Année,Code_INSEE,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,...,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Résultat
0,2022,01001,L'Abergement-Clémenciat,645,108,16.740000,537,83.260000,16,2.480000,...,1,0.16,0.19,520,80.620000,96.830000,21.730000,32.310000,45.960000,droite
1,2012,01001,L'Abergement-Clémenciat,592,84,14.189189,508,85.810811,9,1.520270,...,0,0.00,0.00,499,84.290541,98.228346,30.861723,10.821643,58.316633,droite
2,2012,01002,L'Abergement-de-Varey,215,36,16.744186,179,83.255814,5,2.325581,...,0,0.00,0.00,174,80.930233,97.206704,37.931034,12.643678,49.425287,droite
3,2022,01002,L'Abergement-de-Varey,213,38,17.840000,175,82.160000,3,1.410000,...,1,0.47,0.57,171,80.280000,97.710000,38.600000,35.090000,26.320000,gauche
4,2022,01004,Ambérieu-en-Bugey,8765,2078,23.710000,6687,76.290000,88,1.000000,...,46,0.52,0.69,6553,74.760000,98.000000,34.440000,24.870000,40.680000,droite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70218,2022,ZZ231,Taipei,1709,942,55.120000,767,44.880000,8,0.470000,...,2,0.12,0.26,757,44.290000,98.700000,37.380000,39.230000,23.380000,centre
70219,2022,ZZ233,Nour-Soultan,117,64,54.700000,53,45.300000,2,1.710000,...,0,0.00,0.00,51,43.590000,96.230000,27.450000,50.980000,21.570000,centre
70220,2022,ZZ234,Monterrey,713,553,77.560000,160,22.440000,0,0.000000,...,2,0.28,1.25,158,22.160000,98.750000,20.890000,61.390000,17.720000,centre
70221,2022,ZZ235,Bahamas (Nassau),136,78,57.350000,58,42.650000,3,2.210000,...,0,0.00,0.00,55,40.440000,94.830000,5.450000,45.450000,49.090000,droite


In [11]:
df_election.columns

Index(['Année', 'Code_INSEE', 'Libellé de la commune', 'Inscrits',
       'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs',
       '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot',
       'Exprimés', '% Exp/Ins', '% Exp/Vot', '% gauche/Exp', '% centre/Exp',
       '% droite/Exp', 'Résultat'],
      dtype='object')